<a href="https://colab.research.google.com/github/marborox07-dot/VSGAN-tensorrt-docker/blob/main/WAN2_1_2_2_lightlingvrebuild%E0%B8%99%E0%B8%B2%E0%B8%99_%E0%B8%AB%E0%B8%99%E0%B9%88%E0%B8%AD%E0%B8%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @markdown # 💥1. (NEW Tech)Prepare Environment (Upgraded, Fixed & Fully Compatible with High/Low LoRA)
!pip install -q torchsde sageattention==1.0.6
# --- 1. อัปเกรดความเร็วในการติดตั้งด้วย uv ---
!pip install -q uv
!uv pip install --system -q torch==2.6.0 torchvision==0.21.0
%cd /content
from IPython.display import clear_output
!uv pip install --system -q einops diffusers accelerate xformers==0.0.29.post2 triton==3.2.0
!uv pip install --system -q av spandrel albumentations insightface onnx opencv-python segment_anything ultralytics onnxruntime onnxruntime-gpu imageio
clear_output()

# --- 2. โคลน Repositories หลักและ Custom Nodes ---
!git clone --branch ComfyUI_v0.3.47 https://github.com/Isi-dev/ComfyUI
clear_output()

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
clear_output()

!git clone --branch kjnv1.1.3 https://github.com/Isi-dev/ComfyUI_KJNodes.git
clear_output()

%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!uv pip install --system -q -r requirements.txt
clear_output()

%cd /content
!git clone https://github.com/Isi-dev/Practical-RIFE
%cd /content/Practical-RIFE
!pip install git+https://github.com/rk-exxec/scikit-video.git@numpy_deprecation
!mkdir -p /content/Practical-RIFE/train_log
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/IFNet_HDv3.py -O /content/Practical-RIFE/train_log/IFNet_HDv3.py
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/RIFE_HDv3.py -O /content/Practical-RIFE/train_log/RIFE_HDv3.py
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/refine.py -O /content/Practical-RIFE/train_log/refine.py
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/flownet.pkl -O /content/Practical-RIFE/train_log/flownet.pkl
clear_output()


%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg
clear_output()

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from pathlib import Path
import torch
import numpy as np
import cv2
from PIL import Image
import gc
import sys
import random
import imageio
import subprocess
import shutil
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple, CLIPLoader, CLIPTextEncode, VAEDecode, VAELoader,
    KSampler, KSamplerAdvanced, UNETLoader, LoadImage, SaveImage,
    CLIPVisionLoader, CLIPVisionEncode, LoraLoaderModelOnly, ImageScale
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from custom_nodes.ComfyUI_KJNodes.nodes.model_optimization_nodes import (
    WanVideoTeaCacheKJ, PathchSageAttentionKJ, WanVideoNAG, SkipLayerGuidanceWanVideo
)

from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM
from comfy_extras.nodes_wan import WanImageToVideo
from comfy_extras.nodes_upscale_model import UpscaleModelLoader

def download_with_aria2c(link, folder="/content/ComfyUI/models/loras"):
    import os
    filename = link.split("/")[-1].split("?")[0]
    command = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {link} -d {folder} -o {filename}"
    print("Executing download command:")
    print(command)
    os.makedirs(folder, exist_ok=True)
    get_ipython().system(command)
    return filename

def download_civitai_model(civitai_link, civitai_token, folder="/content/ComfyUI/models/loras"):
    import os
    import time
    os.makedirs(folder, exist_ok=True)
    try:
        model_id = civitai_link.split("/models/")[1].split("?")[0]
    except IndexError:
        raise ValueError("Invalid Civitai URL format.")
    civitai_url = f"https://civitai.com/api/download/models/{model_id}?type=Model&format=SafeTensor"
    if civitai_token:
        civitai_url += f"&token={civitai_token}"
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"model_{timestamp}.safetensors"
    full_path = os.path.join(folder, filename)
    download_command = f"wget --max-redirect=10 --show-progress \"{civitai_url}\" -O \"{full_path}\""
    print("Downloading from Civitai...")
    os.system(download_command)
    local_path = os.path.join(folder, filename)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        print(f"LoRA downloaded successfully: {local_path}")
    else:
        print(f"❌ LoRA download failed: {local_path}")
    return filename

def download_lora(link, folder="/content/ComfyUI/models/loras", civitai_token=None):
    if not link or link.strip() == "":
        return None
    if "civitai.com" in link.lower():
        if not civitai_token:
            raise ValueError("Civitai token is required")
        return download_civitai_model(link, civitai_token, folder)
    else:
        return download_with_aria2c(link, folder)

def model_download(url: str, dest_dir: str, filename: str = None, silent: bool = True) -> bool:
    try:
        Path(dest_dir).mkdir(parents=True, exist_ok=True)
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]
        cmd = ['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M', '-d', dest_dir, '-o', filename, url]
        if silent:
            cmd.extend(['--summary-interval=0', '--quiet'])
            print(f"Downloading {filename}...", end=' ', flush=True)
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if silent: print("Done!")
        else: print(f"Downloaded {filename} to {dest_dir}")
        return filename
    except Exception as e:
        print(f"\nError: {str(e)}")
        return False

# --- UI Forms ---
model_quant = "Q4_K_M" # @param ["Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0"]
lightx2v_rank = "32"

# ช่องดาวน์โหลด LoRA ปกติ
download_loRA_1 = False # @param {type:"boolean"}
lora_1_download_url = "" # @param {"type":"string"}
download_loRA_2 = False # @param {type:"boolean"}
lora_2_download_url = "" # @param {"type":"string"}
download_loRA_3 = False # @param {type:"boolean"}
lora_3_download_url = "" # @param {"type":"string"}

# 🆕 เพิ่มช่องดาวน์โหลดสำหรับ High และ Low Noise LoRA โดยเฉพาะ
download_high_noise_loRA = False # @param {type:"boolean"}
high_noise_lora_url = "" # @param {"type":"string"}
high_noise_lora_strength = 1.00 # @param {type:"number"}

download_low_noise_loRA = False # @param {type:"boolean"}
low_noise_lora_url = "" # @param {"type":"string"}
low_noise_lora_strength = 1.00 # @param {type:"number"}

token_if_civitai_url = "2f4f346c3f4317189c69b9c39db3b238" # @param {"type":"string"}

valid_extensions = {'.safetensors', '.ckpt', '.pt', '.pth', '.sft'}

# จัดการดาวน์โหลด LoRA ปกติ
lora_1 = None
if download_loRA_1 and lora_1_download_url.strip():
    lora_1 = download_lora(lora_1_download_url, civitai_token=token_if_civitai_url)
if lora_1:
    if not any(lora_1.lower().endswith(ext) for ext in valid_extensions):
        print(f"❌ Invalid LoRA 1 format: {lora_1}"); lora_1 = None
    else: clear_output(); print("LoRA 1 downloaded successfully!")

lora_2 = None
if download_loRA_2 and lora_2_download_url.strip():
    lora_2 = download_lora(lora_2_download_url, civitai_token=token_if_civitai_url)
if lora_2:
    if not any(lora_2.lower().endswith(ext) for ext in valid_extensions):
        print(f"❌ Invalid LoRA 2 format: {lora_2}"); lora_2 = None
    else: clear_output(); print("LoRA 2 downloaded successfully!")

lora_3 = None
if download_loRA_3 and lora_3_download_url.strip():
    lora_3 = download_lora(lora_3_download_url, civitai_token=token_if_civitai_url)
if lora_3:
    if not any(lora_3.lower().endswith(ext) for ext in valid_extensions):
        print(f"❌ Invalid LoRA 3 format: {lora_3}"); lora_3 = None
    else: clear_output(); print("LoRA 3 downloaded successfully!")

# 🆕 จัดการดาวน์โหลด High Noise และ Low Noise LoRA
high_noise_lora_file = None
if download_high_noise_loRA and high_noise_lora_url.strip():
    high_noise_lora_file = download_lora(high_noise_lora_url, civitai_token=token_if_civitai_url)
if high_noise_lora_file:
    if not any(high_noise_lora_file.lower().endswith(ext) for ext in valid_extensions):
        print(f"❌ Invalid High Noise LoRA format: {high_noise_lora_file}"); high_noise_lora_file = None
    else: clear_output(); print("💥 High Noise LoRA downloaded successfully!")

low_noise_lora_file = None
if download_low_noise_loRA and low_noise_lora_url.strip():
    low_noise_lora_file = download_lora(low_noise_lora_url, civitai_token=token_if_civitai_url)
if low_noise_lora_file:
    if not any(low_noise_lora_file.lower().endswith(ext) for ext in valid_extensions):
        print(f"❌ Invalid Low Noise LoRA format: {low_noise_lora_file}"); low_noise_lora_file = None
    else: clear_output(); print("🍃 Low Noise LoRA downloaded successfully!")


# ดาวน์โหลด Base Models (High / Low Noise แยกคลัง)
if model_quant == "Q4_K_M":
    dit_model = model_download("https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_high_noise_14B_Q4K_M.gguf", "/content/ComfyUI/models/diffusion_models")
    dit_model2 = model_download("https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_low_noise_14B_Q4K_M.gguf", "/content/ComfyUI/models/diffusion_models")
elif model_quant == "Q5_K_M":
    dit_model = model_download("https://huggingface.co/Isi99999/Wan2.2BasedModels/resolve/main/wan2.2_i2v_high_noise_14B_Q5_K_M.gguf", "/content/ComfyUI/models/diffusion_models")
    dit_model2 = model_download("https://huggingface.co/Isi99999/Wan2.2BasedModels/resolve/main/wan2.2_i2v_low_noise_14B_Q5_K_M.gguf", "/content/ComfyUI/models/diffusion_models")
elif model_quant == "Q6_K":
    dit_model = model_download("https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_high-Q6K.gguf", "/content/ComfyUI/models/diffusion_models")
    dit_model2 = model_download("https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_low-Q6K.gguf", "/content/ComfyUI/models/diffusion_models")
else:
    dit_model = model_download("https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_low_noise_14B_Q8_O.gguf", "/content/ComfyUI/models/diffusion_models")
    dit_model2 = model_download("https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_low_noise_14B_Q8_O.gguf", "/content/ComfyUI/models/diffusion_models")

clear_output()
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/clip_vision_h.safetensors -d /content/ComfyUI/models/clip_vision -o clip_vision_h.safetensors
clear_output()

if lightx2v_rank == "32":
    lightx2v_lora = model_download("https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/lightx2v_I2V_14B_480p_cfg_step_distill_rank32_bf16.safetensors", "/content/ComfyUI/models/loras")
elif lightx2v_rank == "64":
    lightx2v_lora = model_download("https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors", "/content/ComfyUI/models/loras")
else:
    lightx2v_lora = model_download("https://huggingface.co/Isi99999/Wan2.1BasedModels/resolve/main/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank128_bf16.safetensors", "/content/ComfyUI/models/loras")

walkingToViewersL = model_download("https://huggingface.co/Isi99999/Wan2.1_14B-480p_I2V_LoRAs/resolve/main/walking%20to%20viewers_Wan.safetensors", "/content/ComfyUI/models/loras")
walkingFromBehindL = model_download("https://huggingface.co/Isi99999/Wan2.1_14B-480p_I2V_LoRAs/resolve/main/walking_from_behind.safetensors", "/content/ComfyUI/models/loras")
dancingL = model_download("https://huggingface.co/Isi99999/Wan2.1_14B-480p_I2V_LoRAs/resolve/main/b3ll13-d8nc3r.safetensors", "/content/ComfyUI/models/loras")

def upload_file():
    os.makedirs('/content/ComfyUI/input', exist_ok=True)
    uploaded = files.upload()
    paths = []
    for filename in uploaded.keys():
        src_path = f'/content/ComfyUI/{filename}'
        dest_path = f'/content/ComfyUI/input/{filename}'
        shutil.move(src_path, dest_path)
        paths.append(dest_path)
        print(f"File saved to: {dest_path}")
    return paths[0] if paths else None

def upload_fileInt():
    os.makedirs('/content/ComfyUI/output', exist_ok=True)
    uploaded = files.upload()
    paths = []
    for filename in uploaded.keys():
        src_path = f'/content/ComfyUI/{filename}'
        dest_path = f'/content/ComfyUI/output/{filename}'
        shutil.move(src_path, dest_path)
        paths.append(dest_path)
        print(f"File saved to: {dest_path}")
    return paths[0] if paths else None

def extract_frames(video_path, max_frames=None):
    vidcap = cv2.VideoCapture(video_path)
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        success, frame = vidcap.read()
        if not success or (max_frames and len(frames) >= max_frames): break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = torch.from_numpy(frame).float() / 255.0
        frames.append(frame)
    if not frames: return None, fps
    batch = torch.stack(frames, dim=0)
    return batch, fps

def select_every_n_frame_tensor(frames_tensor: torch.Tensor, fps: float, n: int, skip_first: int = 0, max_output_frames: int = 0):
    if frames_tensor is None or frames_tensor.ndim != 4: raise ValueError("frames_tensor must be a 4D tensor of shape (N, H, W, C)")
    if n < 1: raise ValueError("n must be >= 1")
    total_frames = frames_tensor.shape[0]
    if skip_first >= total_frames: return None, 0.0
    frames_to_use = frames_tensor[skip_first:]
    selected_frames = frames_to_use[::n]
    if max_output_frames > 0 and selected_frames.shape[0] > max_output_frames:
        selected_frames = selected_frames[:max_output_frames]
    adjusted_fps = fps / n
    return selected_frames, adjusted_fps

def swapT(pa, f, s):
    if pa == f: pa = s
    return pa

def image_width_height(image):
    if image.ndim == 4: _, height, width, _ = image.shape
    elif image.ndim == 3: height, width, _ = image.shape
    else: raise ValueError(f"Unsupported image shape: {image.shape}")
    return width, height

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames: writer.append_data(frame)
    return output_path

def save_as_mp4U(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"
    frames = []
    for i, img in enumerate(images):
        try:
            if isinstance(img, torch.Tensor): img = img.cpu().numpy()
            if img.max() <= 1.0: img = (img * 255).astype(np.uint8)
            else: img = img.astype(np.uint8)
            if len(img.shape) == 4: img = img[0]
            if len(img.shape) == 3:
                if img.shape[0] in (1, 3, 4): img = np.transpose(img, (1, 2, 0))
                elif img.shape[2] > 4: img = img[:, :, :3]
            elif len(img.shape) == 2: img = np.expand_dims(img, axis=-1)
            frames.append(img)
        except Exception as e:
            print(f"Error processing frame {i}: {str(e)}")
            raise
    try:
        with imageio.get_writer(output_path, fps=fps) as writer:
            for i, frame in enumerate(frames): writer.append_data(frame)
    except Exception as e:
        print(f"Error writing video: {str(e)}")
        raise
    return output_path

def save_as_webp(images, filename_prefix, fps, quality=90, lossless=False, method=4, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webp"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    kwargs = {'fps': int(fps), 'quality': int(quality), 'lossless': bool(lossless), 'method': int(method)}
    with imageio.get_writer(output_path, format='WEBP', mode='I', **kwargs) as writer:
        for frame in frames: writer.append_data(frame)
    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=32, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    kwargs = {'fps': int(fps), 'quality': int(quality), 'codec': str(codec), 'output_params': ['-crf', str(int(quality))]}
    with imageio.get_writer(output_path, format='FFMPEG', mode='I', **kwargs) as writer:
        for frame in frames: writer.append_data(frame)
    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"
    frame = (image.cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(frame).save(output_path)
    return output_path

def save_as_image2(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"
    if isinstance(image, torch.Tensor): image = image.cpu().numpy()
    if image.ndim == 4: image = image[0]
    if image.shape[0] == 3: image = np.transpose(image, (1, 2, 0))
    image = (image * 255).astype(np.uint8)
    Image.fromarray(image).save(output_path)
    return output_path

def upload_image():
    from google.colab import files
    import os
    import shutil
    os.makedirs('/content/ComfyUI/input', exist_ok=True)
    uploaded = files.upload()
    for filename in uploaded.keys():
        src_path = f'/content/ComfyUI/{filename}'
        dest_path = f'/content/ComfyUI/input/{filename}'
        shutil.move(src_path, dest_path)
        return dest_path
    return None

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode
    video_data = open(video_path,'rb').read()
    if video_path.lower().endswith('.mp4'): mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'): mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'): mime_type = "image/webp"
    else: mime_type = "video/mp4"
    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()
    display(HTML(f"""<video width=512 controls autoplay loop><source src="{data_url}" type="{mime_type}"></video>"""))

# Global Variables สำหรับส่งต่อให้เซลล์ 4
global output_path
output_path = ""
output_pathU = ""

def generate_video(
    image_path: str = None,
    LoRA_Strength: float = 1.00,
    rel_l1_thresh: float = 0.275,
    start_percent: float = 0.1,
    end_percent: float = 1.0,
    positive_prompt: str = "fullbody of a female subject in front of a penis, she in a deep throat blowjob,she smashes her head forward and stops there, She swallows the entire penis, her nose smashes agains the man's hips. stops moving. She holds the penis inside her throat. Visible gagging indicates discomfort and nausea during swallowing. (she exhibits strong gag reflex responses, arching her back, shrugging her shoulders and frowning while smashing her nose against the man's hips, she taps with her right hand on the man's hips:1.2)",
    prompt_assist: str = "walking to viewers",
    negative_prompt: str = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺ของม็อด,多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走",
    width: int = 832,
    height: int = 480,
    seed: int = 82628696717253,
    steps: int = 20,
    cfg_scale: float = 1.0,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 33,
    fps: int = 16,
    output_format: str = "mp4",
    overwrite: bool = False,
    use_lora: bool = True,
    use_lora2: bool = True,
    LoRA_Strength2: float = 1.00,
    use_lora3: bool = True,
    LoRA_Strength3: float = 1.00,
    # พารามิเตอร์ของระบบ High / Low Noise LoRA ที่เพิ่มเข้ามาใหม่
    high_noise_lora_file: str = None,
    high_noise_lora_strength: float = 1.00,
    low_noise_lora_file: str = None,
    low_noise_lora_strength: float = 1.00,
    use_lightx2v: bool = False,
    lightx2v_Strength: float = 0.80,
    lightx2v_steps: int = 4,
    use_pusa: bool = False,
    pusa_Strength: float = 1.2,
    pusa_steps: int = 6,
    use_sage_attention: bool = True,
    enable_flow_shift: bool = True,
    shift: float = 8.0,
    enable_flow_shift2: bool = True,
    shift2: float = 8.0,
    end_step1: int = 10,
):
    with torch.inference_mode():
        unet_loader = UnetLoaderGGUF()
        pathch_sage_attention = PathchSageAttentionKJ()
        wan_video_nag = WanVideoNAG()
        teacache = WanVideoTeaCacheKJ()
        model_sampling = ModelSamplingSD3()
        clip_loader = CLIPLoader()
        clip_encode_positive = CLIPTextEncode()
        clip_encode_negative = CLIPTextEncode()
        vae_loader = VAELoader()
        clip_vision_loader = CLIPVisionLoader()
        clip_vision_encode = CLIPVisionEncode()
        load_image = LoadImage()
        wan_image_to_video = WanImageToVideo()
        ksampler = KSamplerAdvanced()
        vae_decode = VAEDecode()
        save_webp = SaveAnimatedWEBP()
        save_webm = SaveWEBM()
        pAssLora = LoraLoaderModelOnly()
        load_lora = LoraLoaderModelOnly()
        load_lora2 = LoraLoaderModelOnly()
        load_lora3 = LoraLoaderModelOnly()
        load_high_noise_lora_node = LoraLoaderModelOnly() # เพิ่มโหนดเฉพาะสำหรับ High Noise LoRA
        load_low_noise_lora_node = LoraLoaderModelOnly()  # เพิ่มโหนดเฉพาะสำหรับ Low Noise LoRA
        load_lightx2v_lora = LoraLoaderModelOnly()
        load_pusa_lora = LoraLoaderModelOnly()
        image_scaler = ImageScale()

        print("Loading Text_Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]
        del clip; torch.cuda.empty_cache(); gc.collect()

        if image_path is None:
            print("Please upload an image file:")
            image_path = upload_image()
        if image_path is None:
            print("No image uploaded!")
            return
        loaded_image = load_image.load_image(image_path)[0]
        width_int, height_int = image_width_height(loaded_image)
        if height == 0: height = int(width * height_int / width_int)

        print(f"Scaling image to {width}x{height}...")
        loaded_image = image_scaler.upscale(loaded_image, "lanczos", width, height, "disabled")[0]
        clip_vision_output = None

        print("Loading VAE...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]
        positive_out, negative_out, latent = wan_image_to_video.encode(positive, negative, vae, width, height, frames, 1, loaded_image, clip_vision_output)
        usedSteps = steps

        # ==========================================
        # 💥 [STAGE 1] HIGH NOISE MODEL PROCESSING
        # ==========================================
        print("Loading high noise Model...")
        model = unet_loader.load_unet(dit_model)[0]
        if enable_flow_shift: model = model_sampling.patch(model, shift)[0]

        if prompt_assist != "none":
            if prompt_assist == "walking to viewers": model = pAssLora.load_lora_model_only(model, walkingToViewersL, 1)[0]
            if prompt_assist == "walking from behind": model = pAssLora.load_lora_model_only(model, walkingFromBehindL, 1)[0]
            if prompt_assist == "b3ll13-d8nc3r": model = pAssLora.load_lora_model_only(model, dancingL, 1)[0]

        # โหลด LoRA ทั่วไป
        if use_lora and lora_1 is not None: model = load_lora.load_lora_model_only(model, lora_1, LoRA_Strength)[0]
        if use_lora2 and lora_2 is not None: model = load_lora2.load_lora_model_only(model, lora_2, LoRA_Strength2)[0]
        if use_lora3 and lora_3 is not None: model = load_lora3.load_lora_model_only(model, lora_3, LoRA_Strength3)[0]

        # 🆕 แทรก High Noise LoRA เข้าไปเฉพาะในโมเดลนี้
        if high_noise_lora_file is not None:
            print(f"🔌 Injecting High Noise LoRA: {high_noise_lora_file} (Strength: {high_noise_lora_strength})")
            model = load_high_noise_lora_node.load_lora_model_only(model, high_noise_lora_file, high_noise_lora_strength)[0]

        if use_lightx2v:
            model = load_lightx2v_lora.load_lora_model_only(model, lightx2v_lora, lightx2v_Strength)[0]
            usedSteps=lightx2v_steps

        if use_sage_attention: model = pathch_sage_attention.patch(model, "auto")[0]
        if rel_l1_thresh > 0: model = teacache.patch_teacache(model, rel_l1_thresh, start_percent, end_percent, "main_device", "14B")[0]

        clear_output()
        print("Generating video with high noise model...")
        sampled = ksampler.sample(model=model, add_noise="enable", noise_seed=seed, steps=usedSteps, cfg=cfg_scale, sampler_name=sampler_name, scheduler=scheduler, positive=positive_out, negative=negative_out, latent_image=latent, start_at_step=0, end_at_step=end_step1, return_with_leftover_noise="enable")[0]
        del model; torch.cuda.empty_cache(); gc.collect()

        # ==========================================
        # 🍃 [STAGE 2] LOW NOISE MODEL PROCESSING
        # ==========================================
        print("Loading low noise Model...")
        model = unet_loader.load_unet(dit_model2)[0]
        if enable_flow_shift2: model = model_sampling.patch(model, shift2)[0]

        if prompt_assist != "none":
            if prompt_assist == "walking to viewers": model = pAssLora.load_lora_model_only(model, walkingToViewersL, 1)[0]
            if prompt_assist == "walking from behind": model = pAssLora.load_lora_model_only(model, walkingFromBehindL, 1)[0]
            if prompt_assist == "b3ll13-d8nc3r": model = pAssLora.load_lora_model_only(model, dancingL, 1)[0]

        # โหลด LoRA ทั่วไป
        if use_lora and lora_1 is not None: model = load_lora.load_lora_model_only(model, lora_1, LoRA_Strength)[0]
        if use_lora2 and lora_2 is not None: model = load_lora2.load_lora_model_only(model, lora_2, LoRA_Strength2)[0]
        if use_lora3 and lora_3 is not None: model = load_lora3.load_lora_model_only(model, lora_3, LoRA_Strength3)[0]

        # 🆕 แทรก Low Noise LoRA เข้าไปเฉพาะในโมเดลเกลี่ยดีเทลนี้
        if low_noise_lora_file is not None:
            print(f"🔌 Injecting Low Noise LoRA: {low_noise_lora_file} (Strength: {low_noise_lora_strength})")
            model = load_low_noise_lora_node.load_lora_model_only(model, low_noise_lora_file, low_noise_lora_strength)[0]

        if use_pusa:
            model = load_pusa_lora.load_lora_model_only(model, lightx2v_lora, pusa_Strength)[0]
            usedSteps=lightx2v_steps

        if use_sage_attention: model = pathch_sage_attention.patch(model, "auto")[0]
        if rel_l1_thresh > 0: model = teacache.patch_teacache(model, rel_l1_thresh, start_percent, end_percent, "main_device", "14B")[0]

        clear_output()
        print("Generating video with low noise model...")
        sampled = ksampler.sample(model=model, add_noise="disable", noise_seed=seed, steps=usedSteps, cfg=cfg_scale, sampler_name=sampler_name, scheduler=scheduler, positive=positive_out, negative=negative_out, latent_image=sampled, start_at_step=end_step1, end_at_step=10000, return_with_leftover_noise="disable")[0]
        del model; torch.cuda.empty_cache(); gc.collect()

        try:
            print("Decoding latents...")
            decoded = vae_decode.decode(vae, sampled)[0]
            del vae; torch.cuda.empty_cache(); gc.collect()

            global output_path
            import datetime
            base_name = "ComfyUI"
            if not overwrite:
                timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                base_name += f"_{timestamp}"

            if frames == 1:
                print("Single frame detected - saving as PNG image...")
                output_path = save_as_image(decoded[0], "ComfyUI")
                display(IPImage(filename=output_path))
            else:
                if output_format.lower() == "webm":
                    print("Saving as WEBM...")
                    output_path = save_as_webm(decoded, base_name, fps=fps, codec="vp9", quality=10)
                elif output_format.lower() == "mp4":
                    print("Saving as MP4...")
                    output_path = save_as_mp4(decoded, base_name, fps)
                else:
                    raise ValueError(f"Unsupported output format: {output_format}")
                display_video(output_path)
        except Exception as e:
            print(f"Error during decoding/saving: {str(e)}")
            raise
        finally:
            clear_memory()

clear_output()
print("✅ Environment Setup & High/Low LoRA Patch Complete!")

✅ Environment Setup & High/Low LoRA Patch Complete!


In [3]:
# @title 📥 Step 2.1: HF High-Speed Downloader (Multi-Links)
# @markdown ### 📌 1. Custom Download Channels
custom_url = "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_high_noise_14B_Q4KM_L.gguf" # @param {type:"string"}
custom_filename = "" # @param {type:"string"}
target_sub_folder = "diffusion_models" # @param [ "checkpoints", "clip", "clip_vision", "configs", "controlnet", "diffusion_models", "embeddings", "gligen", "hypernetworks", "loras", "photomaker", "style_models", "text_encoders", "unet", "upscale_models", "vae", "vae_approx"]

custom2_url = "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_low_noise_14B_Q4KM_L.gguf" # @param {type:"string"}
custom2_filename = "" # @param {type:"string"}
target2_sub_folder = "diffusion_models" # @param [ "checkpoints", "clip", "clip_vision", "configs", "controlnet", "diffusion_models", "embeddings", "gligen", "hypernetworks", "loras", "photomaker", "style_models", "text_encoders", "unet", "upscale_models", "vae", "vae_approx"]

# @markdown ---
# @markdown ### 📝 2. Text Encoder (Auto-routes to text_encoders)
text_encoder_url = "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/umt5_xxl_fp8_e4m3fn_scaled.safetensors" # @param {type:"string"}

# @markdown ---
# @markdown ### 🎨 3. LoRA (Auto-routes to loras)
lora_url_1 = "" # @param {type:"string"}
lora_url_2 = "" # @param {type:"string"}
lora_url_3 = "" # @param {type:"string"}

# @markdown ---
# @markdown ### ✨ 4. VAE & CLIP Vision
vae_url = "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan_2.1_bf16_vae.safetensors" # @param {type:"string"}
clip_vision_url = "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/clip_vision_g.safetensors" # @param {type:"string"}
# @markdown ---

import os
import re
from google.colab import userdata
from huggingface_hub import hf_hub_download

# Enable High-Speed Transfer and Disable Xet for Stability
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
MODELS_DIR = "/content/ComfyUI/models"

# Attempt to retrieve HF Token if available in Colab Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = None

def download_hf_file(url_input, target_folder, manual_filename=""):
    url = url_input.strip()
    if not url:
        return

    clean_repo_id = url
    clean_filename = manual_filename.strip()

    # Extract Repo ID and Path from the HuggingFace URL dynamically
    if "huggingface.co" in url:
        url_match = re.search(r"huggingface\.co/([^/]+/[^/]+)/(?:resolve|blob)/[^/]+/(.+)", url)
        if url_match:
            clean_repo_id = url_match.group(1)
            if not clean_filename:
                clean_filename = url_match.group(2).split("?")[0]

    if clean_repo_id and clean_filename:
        print(f"[*] Source: {clean_repo_id} (File: {clean_filename})")
        print(f"[*] Destination: {target_folder}/")

        dest_path = os.path.join(MODELS_DIR, target_folder)
        os.makedirs(dest_path, exist_ok=True)

        try:
            downloaded_file = hf_hub_download(
                repo_id=clean_repo_id,
                filename=clean_filename,
                local_dir=dest_path,
                local_dir_use_symlinks=False,
                token=HF_TOKEN
            )
            print(f"[+] Successfully downloaded: {downloaded_file}\n")
        except Exception as e:
            print(f"[-] Error downloading {clean_filename}: {e}\n")
    else:
        print(f"[!] Invalid URL format: {url}\n")

# --- Execution Block ---

if custom_url:
    print("--- Processing Custom Link 1 ---")
    download_hf_file(custom_url, target_sub_folder, custom_filename)

if custom2_url:
    print("--- Processing Custom Link 2 ---")
    download_hf_file(custom2_url, target2_sub_folder, custom2_filename)

if text_encoder_url:
    print("--- Processing Text Encoder ---")
    download_hf_file(text_encoder_url, "text_encoders")

lora_urls = [lora_url_1, lora_url_2, lora_url_3]
for i, l_url in enumerate(lora_urls, 1):
    if l_url:
        print(f"--- Processing LoRA {i} ---")
        download_hf_file(l_url, "loras")

if vae_url:
    print("--- Processing VAE ---")
    download_hf_file(vae_url, "vae")

if clip_vision_url:
    print("--- Processing CLIP Vision ---")
    download_hf_file(clip_vision_url, "clip_vision")

clear_output()

In [2]:
import os

# ทำการกำหนดค่าตัวแปรโมเดลด้วยชื่อไฟล์ที่มีอยู่จริงในเครื่อง
dit_model = "wan2.2_i2v_high_noise_14B_Q4KM_L.gguf"
dit_model2 = "wan2.2_i2v_low_noise_14B_Q4KM_L.gguf"

# ตรวจสอบความถูกต้องอีกครั้ง
path1 = os.path.join("/content/ComfyUI/models/diffusion_models", dit_model)
path2 = os.path.join("/content/ComfyUI/models/diffusion_models", dit_model2)

if os.path.exists(path1) and os.path.exists(path2):
    print("✅ เชื่อมต่อตัวแปรกับไฟล์โมเดลสำเร็จ!")
    print(f"High Noise: {dit_model}")
    print(f"Low Noise: {dit_model2}")
    print("\nคุณสามารถไปรันเซลล์ 'Generate Video' ได้เลยครับ")
else:
    print("❌ ยังไม่พบไฟล์ในโฟลเดอร์ กรุณาตรวจสอบตำแหน่งไฟล์อีกครั้ง")

✅ เชื่อมต่อตัวแปรกับไฟล์โมเดลสำเร็จ!
High Noise: wan2.2_i2v_high_noise_14B_Q4KM_L.gguf
Low Noise: wan2.2_i2v_low_noise_14B_Q4KM_L.gguf

คุณสามารถไปรันเซลล์ 'Generate Video' ได้เลยครับ


In [3]:
import os

# ตรวจสอบสถานะไฟล์โมเดลหลัก
print("--- ตรวจสอบไฟล์ในระบบ ---")
models_to_check = {
    "High Noise Model": dit_model,
    "Low Noise Model": dit_model2,
    "Text Encoder": "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "VAE": "wan_2.1_vae.safetensors"
}

missing_found = False
for name, var in models_to_check.items():
    if isinstance(var, bool) and var == False:
        print(f"❌ {name}: ดาวน์โหลดไม่สำเร็จ (ค่าเป็น False)")
        missing_found = True
    elif isinstance(var, str):
        # เช็คว่ามีไฟล์จริงในโฟลเดอร์ไหม
        path_exists = any(os.path.exists(os.path.join(f"/content/ComfyUI/models/{sub}", var))
                         for sub in ["diffusion_models", "text_encoders", "vae"])
        if not path_exists:
            print(f"❌ {name}: ไม่พบไฟล์จริงในระบบ")
            missing_found = True
        else:
            print(f"✅ {name}: พร้อมใช้งาน")

if missing_found:
    print("\n⚠️ คำแนะนำ: ปัญหานี้มักเกิดจากพื้นที่ Disk ใน Colab เต็ม หรือการดาวน์โหลดใน Step 1 หลุด")
    print("กรุณาลองรันเซลล์ 'Prepare Environment' อีกครั้ง หรือตรวจสอบพื้นที่ว่างครับ")

--- ตรวจสอบไฟล์ในระบบ ---
✅ High Noise Model: พร้อมใช้งาน
✅ Low Noise Model: พร้อมใช้งาน
✅ Text Encoder: พร้อมใช้งาน
✅ VAE: พร้อมใช้งาน


In [11]:

# @markdown # 💥2. Upload Image
file_uploaded = upload_image()
display_upload = False # @param {type:"boolean"}
if display_upload:
    if file_uploaded.lower().endswith(('.png', '.jpg', '.jpeg')):
        display(IPImage(filename=file_uploaded))
    else:
        print("Image format cannnot be displayed.")
# @markdown ---

Saving ComfyUI_temp_yvqns_00039_.png to ComfyUI_temp_yvqns_00039_.png


In [12]:
# @markdown # 💥3. Generate Videoของไหม่อาจทำสเกล
import time
start_time = time.time()
# @markdown ### Video Settings
positive_prompt = "being penetrated vaginally by one man while another man holds her down and lifts her up. The scene shows the man's penis repeatedly entering and exiting her vagina, accompanied by graphic ejaculation. This scene captures explicit sexual activity with realistic detail, focusing on penetration, penis movement, and the woman's body responses. Skin texture, lighting, and physical contact are rendered with high realism, emphasizing intercourse and ejaculation through a condom." # @param {"type":"string"}
prompt_assist = "none" # @param ["none","walking to camera", "walking from camera", "swaying"]
prompt_assist = swapT(prompt_assist, "walking to camera", "walking to viewers")
prompt_assist = swapT(prompt_assist, "walking from camera", "walking from behind")
prompt_assist = swapT(prompt_assist, "swaying", "b3ll13-d8nc3r")
positive_prompt = f"{positive_prompt} {prompt_assist}." if prompt_assist != "none" else positive_prompt

negative_prompt = "worst quality, low quality, bad anatomy, bad hands, text, error, missing fingers, extra digit, fewer digits, cropped, jpeg artifacts, signature, watermark, username, blurry" # @param {"type":"string"}

# --- [ตั้งค่าความละเอียด] ---
width = 720 # @param {"type":"number"}
height = 380 # @param {"type":"number"}
resolution_scale = "1.0" # @param ["0.5", "0.75", "1.0", "1.25", "1.5", "2.0"] {allow-input: true}

# --- [จุดอัปเกรด: เพิ่มแผงคุมสุ่ม Seed] ---
seed = 42 # @param {"type":"integer"}
randomize_seed = True # @param {type:"boolean"}

high_noise_steps = 2 # @param {"type":"integer", "min":1, "max":25}
steps = 4 # @param {"type":"integer", "min":1, "max":50}
cfg_scale = 1 # @param {"type":"number", "min":1, "max":20}
sampler_name = "euler" # @param ["uni_pc", "uni_pc_bh2", "ddim","euler", "euler_cfg_pp", "euler_ancestral", "euler_ancestral_cfg_pp", "heun", "heunpp2","dpm_2", "dpm_2_ancestral","lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral", "dpmpp_2s_ancestral_cfg_pp", "dpmpp_sde", "dpmpp_sde_gpu","dpmpp_2m", "dpmpp_2m_cfg_pp", "dpmpp_2m_sde", "dpmpp_2m_sde_gpu", "dpmpp_3m_sde", "dpmpp_3m_sde_gpu", "ddpm", "lcm","ipndm", "ipndm_v", "deis", "res_multistep", "res_multistep_cfg_pp", "res_multistep_ancestral", "res_multistep_ancestral_cfg_pp","gradient_estimation", "er_sde", "seeds_2", "seeds_3"]
scheduler = "simple" # @param ["simple","normal","karras","exponential","sgm_uniform","ddim_uniform","beta","linear_quadratic","kl_optimal","uni_pc","uni_pc_bh2"]

frames = 150 # @param {"type":"integer", "min":1, "max":120}
fps = 12
output_format = "mp4"
overwrite_previous_video = True # @param {type:"boolean"}

# @markdown ### Model Configuration
use_sage_attention = True # @param {type:"boolean"}
use_flow_shift = True # @param {type:"boolean"}
flow_shift = 8 # @param {"type":"slider","min":0.0,"max":100.0,"step":0.01}
flow_shift2 = 8 # @param {"type":"slider","min":0.0,"max":100.0,"step":0.01}

# @markdown ### Noise LoRA Strength Control
high_noise_lora_strength = 1 # @param {"type":"slider","min":-10.0,"max":10.0,"step":0.01}
low_noise_lora_strength = 1 # @param {"type":"slider","min":-10.0,"max":10.0,"step":0.01}

# @markdown ### Wan2.1 Based Models LoRA Configuration
use_lightx2v = False # @param {type:"boolean"}
lightx2v_Strength = 1.0 # @param {"type":"slider","min":-10,"max":10,"step":0.01}
use_lightx2v2 = False # @param {type:"boolean"}
lightx2v2_Strength = 0.2 # @param {"type":"slider","min":-10,"max":10,"step":0.01}

# @markdown ### LoRA Configuration
use_lora = False # @param {type:"boolean"}
LoRA_Strength = 0.45 # @param {"type":"slider","min":-100,"max":100,"step":0.01}
use_lora2 = False # @param {type:"boolean"}
LoRA_Strength2 = 1 # @param {"type":"slider","min":-100,"max":100,"step":0.01}
use_lora3 = False # @param {type:"boolean"}
LoRA_Strength3 = 1 # @param {"type":"slider","min":-100,"max":100,"step":0.01}

# @markdown ### Teacache Settings
rel_l1_thresh = 0.0 # @param {"type":"slider","min":0.0,"max":10,"step":0.001}
start_percent = 0 # @param {"type":"slider","min":0.0,"max":1.0,"step":0.01}
end_percent = 1 # @param {"type":"slider","min":0.0,"max":1.0,"step":0.01}

# @markdown ---

import random
# --- [ลอจิกประมวลผลการสุ่มค่า Seed] ---
if randomize_seed:
    current_seed = random.randint(0, 2**32 - 1)
    print(f"🎲 Randomize Seed Enabled! Generated New Seed: {current_seed}")
else:
    current_seed = seed
    print(f"🔒 Fixed Seed Enabled! Using: {current_seed}")

# --- [จุดคำนวณปรับสเกลความละเอียด] ---
scale_factor = float(resolution_scale)
calculated_width = int((width * scale_factor) // 16) * 16
calculated_height = int((height * scale_factor) // 16) * 16
print(f"📐 Resolution Scale: {resolution_scale} -> Rendered Size: {calculated_width}x{calculated_height}")

import os
output_dir = "/content/ComfyUI/output"
os.makedirs(output_dir, exist_ok=True)
oIoutput_path = os.path.join(output_dir, f"wan_generation_{current_seed}.mp4")

# with torch.inference_mode():
generate_video(
    image_path=file_uploaded,
    LoRA_Strength=LoRA_Strength,
    high_noise_lora_strength=high_noise_lora_strength,
    low_noise_lora_strength=low_noise_lora_strength,
    rel_l1_thresh=rel_l1_thresh,
    start_percent=start_percent,
    end_percent = end_percent,
    positive_prompt=positive_prompt,
    prompt_assist=prompt_assist,
    negative_prompt=negative_prompt,
    width=calculated_width,
    height=calculated_height,
    seed=current_seed, # สลับมาส่งค่าพารามิเตอร์ seed ตัวแปรล่าสุดที่ผ่านระบบตรวจจับสุ่มแล้ว
    steps=steps,
    cfg_scale=cfg_scale,
    sampler_name=sampler_name,
    scheduler=scheduler,
    frames=frames,
    fps=fps,
    output_format=output_format,
    overwrite=overwrite_previous_video,
    use_lora = use_lora,
    use_lora2=use_lora2,
    LoRA_Strength2=LoRA_Strength2,
    use_lora3=use_lora3,
    LoRA_Strength3=LoRA_Strength3,
    use_lightx2v=use_lightx2v,
    lightx2v_Strength=lightx2v_Strength,
    lightx2v_steps=steps,
    use_pusa=use_lightx2v2,
    pusa_Strength=lightx2v2_Strength,
    pusa_steps=steps,
    use_sage_attention = use_sage_attention,
    enable_flow_shift = use_flow_shift,
    shift = flow_shift,
    enable_flow_shift2 = use_flow_shift,
    shift2 = flow_shift2,
    end_step1 = high_noise_steps
)

import glob
video_files = glob.glob(os.path.join(output_dir, "*.mp4"))
if video_files:
    oIoutput_path = max(video_files, key=os.path.getctime)

end_time = time.time()
duration = end_time - start_time
mins, secs = divmod(duration, 60)
print(f"Final Seed Used: {current_seed}")
print(f"✅ Generation completed in {int(mins)} min {secs:.2f} sec")

clear_memory()

Generating video with low noise model...
Attempting to release mmap (215)
Patching comfy attention to use sageattn


  0%|          | 0/2 [00:00<?, ?it/s]

Restoring initial comfy attention
Decoding latents...
Saving as MP4...


Final Seed Used: 3032725959
✅ Generation completed in 11 min 30.35 sec


In [10]:

# @markdown # 💥4. Apply Frame Interpolation
interpolate_optional_video=True # @param {type:"boolean"}

if interpolate_optional_video:
    try:
        output_path = oIoutput_path
    except NameError:
        pass


import glob
from IPython.display import Video as outVid
import time
start_time = time.time()

FRAME_MULTIPLIER = 2 # @param {"type":"number"}
vid_fps = 30 # @param {"type":"number"}
crf_value = 20 # @param {"type":"slider","min":0,"max":51,"step":1}

print(f"Converting video to {vid_fps} fps...")

%cd /content/Practical-RIFE

# Suppress ALSA errors
os.environ["XDG_RUNTIME_DIR"] = "/tmp"
os.environ["SDL_AUDIODRIVER"] = "dummy"

# Disable warnings from ffmpeg about missing audio
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"
os.environ["FFMPEG_LOGLEVEL"] = "quiet"

!python3 inference_video.py --multi={FRAME_MULTIPLIER} --fps={vid_fps} --video={output_path} --scale={1}
video_folder = "/content/ComfyUI/output/"

# Find the latest MP4 file
video_files = glob.glob(os.path.join(video_folder, "*.mp4"))

if video_files:
    latest_video = max(video_files, key=os.path.getctime)
    # !ffmpeg -i "{latest_video}" -vcodec libx264 -crf 18 -preset fast output_converted.mp4 -loglevel error -y
    !ffmpeg -i "{latest_video}" -vcodec libx264 -crf {crf_value} -preset fast output_converted.mp4 -loglevel error -y

    print(f"Displaying video: {latest_video}")
    # display(outVid("output_converted.mp4", embed=True))
    display_video("output_converted.mp4")
    # displayVid(outVid(latest_video, embed=True))
else:
    print("❌ No video found in output/")

del video_files

end_time = time.time()
duration = end_time - start_time
mins, secs = divmod(duration, 60)
print(f"✅ Frame Interpolation completed in {int(mins)} min {secs:.2f} sec")

clear_memory()

%cd /content/ComfyUI

# @markdown ---

Converting video to 30 fps...
/content/Practical-RIFE
Loaded 3.x/4.x HD model.
/content/ComfyUI/output/ComfyUI.mp4, 149.0 frames in total, 12.0FPS to 30FPS
Will not merge audio because using png or fps flag!
 99% 148/149.0 [00:04<00:00, 31.91it/s]
Displaying video: /content/ComfyUI/output/ComfyUI_2X_30fps.mp4


✅ Frame Interpolation completed in 0 min 21.46 sec
/content/ComfyUI
